In [ ]:
#@markdown # Imports
#!uv pip install xmltodict html-to-markdown[lxml,html5lib] seaborn

import os
import time
import json
import pickle
import requests
import xmltodict
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.cli import tqdm
from bs4 import BeautifulSoup
from multiprocessing import Pool
from urllib.parse import urljoin, urlencode
from html_to_markdown import convert_to_markdown
from IPython.display import Markdown, display
from google.colab import userdata
from typing import Literal, List, Optional
from pydantic import BaseModel, Field
from openai import OpenAI
import hashlib
from pathlib import Path

tqdm.pandas()

os.chdir('/projects/moties')

if 'api_key' not in globals():
  api_key = userdata.get('RUGLLM')
base_url="https://llm.hpc.rug.nl/v1"

check_cache = True #@param {"type": "boolean"}

client = OpenAI(api_key=api_key, base_url=base_url)

model='mistralai/Mistral-Small-3.2-24B-Instruct-2506'

CACHE_DIR = Path(".cache_responses")
CACHE_DIR.mkdir(exist_ok=True)


In [1216]:
#@markdown # Load, process and merge LLM annotations and votes.

# Load data from `Fetch Stemmingen`.
with open('decision_data.p3', 'rb') as f:
  data = pickle.load(f)
besluiten = data['besluiten'].rename({'Id': 'Besluit_Id'}, axis=1).set_index('Besluit_Id')
zaken = data['zaken']
documenten = data['documenten']
versies = data['versies']
stemmen = data['stemmen']
texts = data['texts']
zaken_ = zaken.set_index('Besluit_Id')[[
  'Nummer', 'Soort', 'Titel', 'Citeertitel', 'Alias', 'Onderwerp', 'GestartOp',
  'Organisatie', 'Grondslagvoorhang', 'Termijn', 'Vergaderjaar', 'Volgnummer',
  'HuidigeBehandelstatus', 'Afgedaan', 'Id']].rename({'Id': 'Zaak_Id'}, axis=1)
zaken__ = pd.concat([besluiten, zaken_], axis=1)

# Only parties with seats, ordered by seats
parties = list(stemmen.T.groupby('ActorFractie').sum().sum(1).sort_values(ascending=False).index)
party_selection = ['JA21', 'Volt', 'FVD', 'DENK', 'PvdD', 'SGP', 'ChristenUnie', 'SP', 'CDA', 'BBB', 'D66', 'NSC', 'VVD', 'GroenLinks-PvdA', 'PVV']
parties = [x for x in parties if x in party_selection]

# Fetch all LLm responses, go from decision/motion text -> annotations to Besluit_Id -> annotations,
# i.e. link LLM annotations to votes.
responses = {}
for fn in os.listdir(CACHE_DIR):
  with open(CACHE_DIR / fn) as f:
    data = json.load(f)
    text = data['messages'][1]['content']
    try:
      response = json.loads(data['response'])
      responses[text] = response
    except json.JSONDecodeError: pass
responses_ = texts.apply(responses.get).apply(pd.Series)

# Make zaken to besluiten
zaak_id_to_besluit_id = {v:k for k, v in zaken__['Zaak_Id'].items()}
responses_['Besluit_Id'] = responses_.index.to_series().apply(zaak_id_to_besluit_id.get)
responses__ = responses_.reset_index(drop=True).set_index('Besluit_Id')
responses__ = responses__[responses__['doel'].notna()].copy()

# If both voor+tegen are n.v.t., ignore the entire thing. This is 99.9% of the n.v.t.'s
# This is usually something like effect on housing on a motion about animal wellfare
for c in stance_columns:
  q = (responses__[f'{c}_van_stem_voor'] == 'n.v.t.') & (responses__[f'{c}_van_stem_tegen'] == 'n.v.t.')
  responses__.loc[q, f'{c}_van_stem_tegen'] = 'ignore'
  responses__.loc[q, f'{c}_van_stem_voor'] = 'ignore'

# eu_kaders is not thrustworthily annotated, so it will be removed in the webapp
# responses__['eu_kaders'].replace('geen / niet relevant', 'geen conflict', inplace=True)
# responses__['eu_kaders'].replace('n.v.t.', 'geen conflict', inplace=True)

# Some votes from the previous period are modified after December 6th 2023,
# so the date isn't trustworthy. So delete any where GL or PvdA exists,
# as they were only there in the previous period.
stemmen = stemmen.reset_index(1, drop=True).loc[responses__.index]
stemmen = stemmen[stemmen['GroenLinks'].sum(1) == 0]
stemmen = stemmen[stemmen['PvdA'].sum(1) == 0]
stemmen = stemmen[parties]

# merge all.
stemmen_ = pd.concat([
    pd.concat([pd.concat([
        zaken__.loc[stemmen.index], responses__.loc[stemmen.index]], axis=1)
    ], keys=['metadata'], axis=1),
    stemmen
], axis=1)

In [1214]:
#@markdown # process votes to their impact, e.g. lower economy, more taxes, etc

vote_map = {'Voor': '_van_stem_voor', 'Tegen': '_van_stem_tegen'}
stance_columns = [
 'asiel_toegankelijkheid', 'box3_effect', 'fiscaal_label',
 'defensieuitgaven', 'dierenwelzijn_effect',
 'economische_kosteneffect',
 'gemeentelijke_last', 'huurmarkt_effect', 'hypotheeklasten_effect',
 'israel_effect', 'kinderopvang_betaalbaarheid',
 'koopwoning_effect', 'kosten_van_leven_effect', 'milieu_effect',
 'oekraine_effect', 'palestina_effect', 'pas_melders_effect',
 'provinciale_last', 'schiphol_capaciteit', 'zorg_effect',
 'sociale_zekerheidseffect', 'veiligheids_effect',
 'mensenrechten_effect', ]

drop_columns = ['Alias', 'Grondslagvoorhang', 'HuidigeBehandelstatus',
 'AgendapuntZaakBesluitVolgorde', 'Agendapunt_Id', 'ApiGewijzigdOp',
 'Verwijderd', 'Status', 'Opmerking', 'BesluitTekst',
 'GewijzigdOp', 'Status', 'StemmingsSoort', 'BesluitSoort',

 'asiel_toegankelijkheid_van_stem_tegen', 'asiel_toegankelijkheid_van_stem_voor',
 'box3_effect_van_stem_tegen', 'box3_effect_van_stem_voor',
 'coalitieakkoord_consistentie_van_stem_tegen', 'coalitieakkoord_consistentie_van_stem_voor',
 'defensieuitgaven_van_stem_tegen', 'defensieuitgaven_van_stem_voor',
 'dierenwelzijn_effect_van_stem_tegen', 'dierenwelzijn_effect_van_stem_voor',
 'economische_kosteneffect_van_stem_tegen', 'economische_kosteneffect_van_stem_voor',

 'financieringsbron_van_stem_tegen', 'financieringsbron_van_stem_voor',
 'fiscaal_label_van_stem_tegen', 'fiscaal_label_van_stem_voor',
 'gemeentelijke_last_van_stem_tegen', 'gemeentelijke_last_van_stem_voor',
 'huurmarkt_effect_van_stem_tegen', 'huurmarkt_effect_van_stem_voor',
 'hypotheeklasten_effect_van_stem_tegen', 'hypotheeklasten_effect_van_stem_voor',
 'israel_effect_van_stem_tegen', 'israel_effect_van_stem_voor',

 'kinderopvang_betaalbaarheid_van_stem_tegen', 'kinderopvang_betaalbaarheid_van_stem_voor',
 'koopwoning_effect_van_stem_tegen', 'koopwoning_effect_van_stem_voor',
 'kosten_van_leven_effect_van_stem_tegen', 'kosten_van_leven_effect_van_stem_voor',
 'milieu_effect_van_stem_tegen', 'milieu_effect_van_stem_voor',
 'oekraine_effect_van_stem_tegen', 'oekraine_effect_van_stem_voor',
 'palestina_effect_van_stem_tegen', 'palestina_effect_van_stem_voor',
 'pas_melders_effect_van_stem_tegen', 'pas_melders_effect_van_stem_voor',
 'provinciale_last_van_stem_tegen', 'provinciale_last_van_stem_voor',
 'schiphol_capaciteit_van_stem_tegen', 'schiphol_capaciteit_van_stem_voor',
 'sociale_zekerheidseffect_van_stem_tegen', 'sociale_zekerheidseffect_van_stem_voor',

 'veiligheids_effect_van_stem_tegen', 'veiligheids_effect_van_stem_voor',
 'zorg_effect_van_stem_tegen', 'zorg_effect_van_stem_voor',
  'mensenrechten_effect_van_stem_voor', 'mensenrechten_effect_van_stem_tegen'
]
keep_columns = [
  'bevat_kostenstrategie', 'notities', 'bronnen', 'tijdshorizon', 'onderwerp',
  'samenvatting_van_besluit', 'uitvoerende_instanties', 'doel',
  'begunstigden_van_stem_voor', 'begunstigden_van_stem_tegen',
  'Titel', 'Onderwerp', 'Soort', 'Citeertitel',
  'GestartOp', 'Organisatie', 'Termijn', 'Vergaderjaar', 'Volgnummer',
  'Afgedaan', 'Nummer', 'Zaak_Id',
  'coalitieakkoord_consistentie', 'eu_kaders', 'uitvoeringsmoeilijkheid',
  'financieringsbron',

  'pas_melders_effect_notities', 'box3_effect_notities', 'veiligheids_effect_notities',
  'juridische_risico_notities', 'oekraine_effect_notities', 'kinderopvang_betaalbaarheid_notities',
  'dierenwelzijn_effect_notities', 'fiscaal_label_notities', 'kosten_van_leven_notities',
  'hypotheeklasten_notities', 'israel_effect_notities', 'provinciale_last_notities',
  'milieu_effect_notities', 'uitvoeringsmoeilijkheid_notities', 'zorg_effect_notities',
  'palestina_effect_notities', 'coalitieakkoord_consistentie_notities',
  'financieringsbron_notities', 'tijdshorizon_notities', 'economische_kosteneffect_notities',
  'eu_kaders_notities', 'gemeentelijke_last_notities', 'mensenrechten_effect_notities',
  'huurmarkt_effect_notities',  'schiphol_capaciteit_notities', 'juridische_risico',
  'koopwoning_effect_notities', 'defensieuitgaven_notities', 'asiel_toegankelijkheid_notities',
  'sociale_zekerheidseffect_notities',
]

print(set(stemmen_['metadata'].columns) - set(drop_columns) - set(keep_columns))

def process(sample, besluit_id):
  return [{
    'Besluit_Id': besluit_id,
    **sample['metadata'][keep_columns].to_dict(),
    **{c: (
      sample['metadata'][f'{c}_van_stem_{vote.lower()}'] if vote in {'Voor', 'Tegen'} else 'not-participated'
    ) for c in stance_columns},
    'begunstigden': sample['metadata'][f'begunstigden_van_stem_{vote.lower()}'] if vote in {'Voor', 'Tegen'} else [],
    'count': int(count),
    'party': party,
    'vote': vote
    }
    for (party, vote), count in sample[parties].items()
    if count > 0
  ]

with Pool(20) as pool:
  promises = stemmen_.apply(lambda sample: pool.apply_async(process, args=(sample.copy(), sample.name)), axis=1)
  party_impacts = pd.DataFrame([
    impact for promise in tqdm(promises) for impact in promise.get()])

party_impacts['begunstigden'] = party_impacts['begunstigden'].apply(lambda x: [] if x!=x else [x.lower() for x in x])

set()


100%|██████████| 7808/7808 [00:18<00:00, 422.25it/s]


In [1215]:
#@markdown # Split into metadata not dependent on votes, and voting impacts. Export JS-friendly and compact.

ps = party_impacts

responses__['Nummer'] = responses__.index.to_series().apply(lambda x: zaken__.loc[x, 'Nummer'])
impacts__ = responses__.set_index('Nummer').apply(lambda x: pd.Series({c: [x[c+'_van_stem_tegen'], x[c+'_van_stem_voor']] for c in stance_columns}), axis=1)

drop_columns = ['Alias', 'Grondslagvoorhang', 'HuidigeBehandelstatus']
separate = [
  'bevat_kostenstrategie', 'notities', 'bronnen', 'tijdshorizon', 'onderwerp',
  'samenvatting_van_besluit', 'uitvoerende_instanties', 'doel',
  'begunstigden_van_stem_voor', 'begunstigden_van_stem_tegen',
  'Titel', 'Onderwerp', 'Soort', 'Citeertitel',
  'GestartOp', 'Organisatie', 'Termijn', 'Vergaderjaar', 'Volgnummer',
  'Afgedaan', # 'Nummer', 'Zaak_Id',
  'coalitieakkoord_consistentie', 'eu_kaders', 'uitvoeringsmoeilijkheid', 'financieringsbron',

  'pas_melders_effect_notities', 'box3_effect_notities', 'veiligheids_effect_notities',
  'juridische_risico_notities', 'oekraine_effect_notities', 'kinderopvang_betaalbaarheid_notities',
  'dierenwelzijn_effect_notities', 'fiscaal_label_notities', 'kosten_van_leven_notities',
  'hypotheeklasten_notities', 'israel_effect_notities', 'provinciale_last_notities',
  'milieu_effect_notities', 'uitvoeringsmoeilijkheid_notities', 'zorg_effect_notities',
  'palestina_effect_notities', 'coalitieakkoord_consistentie_notities',
  'financieringsbron_notities', 'tijdshorizon_notities', 'economische_kosteneffect_notities',
  'eu_kaders_notities', 'gemeentelijke_last_notities', 'mensenrechten_effect_notities',
  'huurmarkt_effect_notities',  'schiphol_capaciteit_notities', 'juridische_risico',
  'koopwoning_effect_notities', 'defensieuitgaven_notities', 'asiel_toegankelijkheid_notities',
  'sociale_zekerheidseffect_notities',
]

expanded_columns = [c for c in ps.columns if c not in separate and c not in drop_columns]
metadata = pd.concat([ps.groupby(['Nummer'])[separate].first(), impacts__], axis=1).reset_index()
metadata = {k: v for k, v in metadata.to_dict(orient='split').items() if k in {'columns', 'data'}}

data = {k: v for k, v in ps[expanded_columns].to_dict(orient='split').items() if k in {'columns', 'data'}}
data['data'] = [[None if type(x) != list and pd.isna(x) else x for x in x] for x in data['data']]
data['metadata'] = [[None if type(x) != list and pd.isna(x) else x for x in x] for x in metadata['data']]
data['metadata_columns'] = metadata['columns']

with open('party_stances.json', 'w') as f:
  json.dump(data, f)

from google.colab.files import download
download('party_stances.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>